# Movie `reviews_1000`：Kaggle 预生成 KV cache

调用 **`pregenerate_movie_kv_caches.py`**，默认 **`ExpectedAttentionPress`** 在 prefill 写出每条样本的 `DynamicCache`（`.pt`）。

## Finch with / without CPT 对齐

| 模式 | 写入 context（经 `Review:` 前缀包装前） |
|------|----------------------------------------|
| **without CPT** | `review` + 空行 + 情感分类任务句（question） |
| **with CPT** | 上式后再接 `CPT:` + 摘要（行内 `cpt` 非空时） |

本 Notebook 使用 **`--run-both-cpt-modes`**：每个压缩率依次跑两种；若 CSV 无 `cpt` 列则自动跳过 with-CPT。

## Checkpoint

**`--checkpoint-csv`**：每个 `(compression_ratio, use_cpt)` 批次结束追加一行，`status=ok` 的整批下次跳过；批内中断靠已生成的 `.pt` 自动 `skip`。

## 数据与代码

1. **Add data**：`reviews_1000.csv`。
2. **仓库**：挂载本仓库 zip，或下方 **`CLONE_REPO=True`**（需 Internet）。
3. **HF**：Secrets → **`HF_TOKEN`**。
4. **持久化**：Kaggle 默认 **`/kaggle/working`**；若你在 **Colab** 跑可把 `MY_DRIVE_ROOT` 设为 `Path("/content/drive/MyDrive")`。最后一格可打 zip 从 Output 下载。

> 预生成 KV 与 **Finch** 压缩器不同；若解码阶段必须用 Finch 压出的 cache，需另行扩展脚本 `--press`（当前未在 pipeline 上验证 Finch）。


In [ ]:
!pip install -q "kvpress>=0.2" "transformers>=4.45" accelerate pandas tqdm safetensors sentencepiece protobuf

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ---------- Hugging Face ----------
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        try:
            t = _sec.get_secret(name)
            if t:
                os.environ["HF_TOKEN"] = t
                os.environ["HUGGING_FACE_HUB_TOKEN"] = t
                print("Loaded secret:", name)
                break
        except Exception:
            continue
except Exception as e:
    print("Secrets:", e)

# ---------- 持久目录：Google Drive 或其它已挂载路径 ----------
# Kaggle 无 Colab 式一键 Drive。任选其一：
# 1) 在「本机 / 其它环境」跑：把 MY_DRIVE_ROOT 设为你的 Google Drive 根（已挂载）。
# 2) 在 Kaggle：默认写 /kaggle/working；跑完后用下一格打包下载，或再用 kaggle datasets version 推到 Dataset。
# 3) 若你自行用 fuse/rclone 把 Drive 挂到某路径，把 MY_DRIVE_ROOT 改成该路径（例如 Path("/kaggle/working/gdrive/MyDrive")）。
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    MY_DRIVE_ROOT = Path("/content/drive/MyDrive")
    print("Colab: Google Drive 已挂载", MY_DRIVE_ROOT)
except ImportError:
    MY_DRIVE_ROOT = None
# Kaggle 等：若自行挂载 Drive，取消注释并改路径
# MY_DRIVE_ROOT = Path("/kaggle/working/gdrive/MyDrive")

REL = Path("kv-compression-benchmark/movie_kv_pregen")

if MY_DRIVE_ROOT is not None and MY_DRIVE_ROOT.is_dir():
    base = MY_DRIVE_ROOT / REL
    CACHE_DIR = base / "caches"
    CHECKPOINT_CSV = base / "movie_kv_pregen_checkpoint.csv"
    print("Persistence: Drive (or custom mount) ->", base)
else:
    base = Path("/kaggle/working/movie_kv_pregen")
    CACHE_DIR = base / "caches"
    CHECKPOINT_CSV = base / "movie_kv_pregen_checkpoint.csv"
    print("Persistence: /kaggle/working (会话结束会清空，请及时下载或同步到 Dataset)")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CSV.parent.mkdir(parents=True, exist_ok=True)

# ---------- 运行参数 ----------
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
RATIOS = ["0.0", "0.2", "0.4", "0.5", "0.6", "0.8"]
MAX_ROWS = None
PRESS = "expected_attention"
EXTRA_ARGS: list[str] = []

CLONE_REPO = False
BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
WORK_ROOT = Path("/kaggle/working")

print("CHECKPOINT_CSV:", CHECKPOINT_CSV.resolve())
print("CACHE_DIR:", CACHE_DIR.resolve())


In [ ]:
from __future__ import annotations

from pathlib import Path

MARKER = Path("benchmarks/kv_cache_pregen/pregenerate_movie_kv_caches.py")


def find_reviews_csv() -> Path:
    for p in Path("/kaggle/input").rglob("reviews_1000.csv"):
        return p
    raise FileNotFoundError("Add Data: 挂载含 reviews_1000.csv 的 Dataset。")


def find_repo_root() -> Path:
    if CLONE_REPO:
        dest = WORK_ROOT / "kv-compression-benchmark"
        if not (dest / MARKER).is_file():
            import subprocess
            subprocess.check_call(["git", "clone", "--depth", "1", BENCH_URL, str(dest)])
        return dest
    roots = list(Path("/kaggle/input").iterdir())
    for root in roots:
        if (root / MARKER).is_file():
            return root
        if root.is_dir():
            for sub in root.iterdir():
                if sub.is_dir() and (sub / MARKER).is_file():
                    return sub
    raise FileNotFoundError(
        "未找到仓库。请挂载含 benchmarks/kv_cache_pregen/ 的 zip，或设 CLONE_REPO=True（需 Internet）。"
    )


CSV_PATH = find_reviews_csv()
REPO_ROOT = find_repo_root()
SCRIPT = REPO_ROOT / MARKER

print("CSV:", CSV_PATH)
print("REPO_ROOT:", REPO_ROOT)
print("SCRIPT:", SCRIPT)


In [ ]:
import shlex

cmd = [
    sys.executable,
    str(SCRIPT),
    "--csv",
    str(CSV_PATH),
    "--cache-dir",
    str(CACHE_DIR),
    "--model",
    MODEL,
    "--compression-ratio",
    *RATIOS,
    "--press",
    PRESS,
    "--device-map",
    "auto",
    "--bf16",
    "--no-flash-attn",
    "--run-both-cpt-modes",
    "--checkpoint-csv",
    str(CHECKPOINT_CSV),
]
if MAX_ROWS is not None:
    cmd += ["--max-rows", str(int(MAX_ROWS))]
cmd += EXTRA_ARGS

print("Command:", shlex.join(cmd))
subprocess.check_call(cmd, cwd=str(REPO_ROOT), env=os.environ.copy())
print("Done.")


In [ ]:
# 打包 caches + checkpoint，便于从 Output 下载或再上传到 Kaggle Dataset
import shutil
from pathlib import Path

bundle = Path("/kaggle/working/movie_kv_pregen_bundle")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)
if CACHE_DIR.is_dir():
    shutil.copytree(CACHE_DIR, bundle / "caches", dirs_exist_ok=True)
if CHECKPOINT_CSV.is_file():
    shutil.copy2(CHECKPOINT_CSV, bundle / CHECKPOINT_CSV.name)

out_zip = Path("/kaggle/working/movie_kv_pregen_bundle.zip")
shutil.make_archive(str(out_zip.with_suffix("")), "zip", str(bundle.parent), bundle.name)
print("Wrote:", out_zip)
